# Time series fundamentals

Everything so far assumed rows were **independent** and interchangeable. Time
series break that: the *order* carries information, and two assumptions from the
[Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter now fail.

- **Random train/test splitting leaks the future** into training.
- **K-fold needs a time-aware variant** (rolling-origin / expanding window).

This is "a different data shape" — we treat it on its own terms.

In [ ]:
// A synthetic daily series: upward trend + weekly (period-7) seasonality, plus a
// little reproducible noise (real series are never perfectly clean — and a purely
// deterministic series breaks statistical tests like the ADF check below). This is
// the same series the forecasting chapter uses.
let series: Vec<f64> = {
    let mut seed = 12345u64;
    (0..56).map(|t| {
        seed = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
        let noise = ((seed >> 11) as f64 / (1u64 << 53) as f64 - 0.5) * 1.4;
        let trend = 10.0 + 0.2 * t as f64;
        let weekly = 3.0 * ((t as f64) * 2.0 * std::f64::consts::PI / 7.0).sin();
        trend + weekly + noise
    }).collect()
};
let split = (series.len() as f64 * 0.8) as usize;

// Chronological split: train on the EARLIER data, test on the later.
{
    let (train, test) = series.split_at(split);
    println!("series = {} points; train = {}, test = {}", series.len(), train.len(), test.len());
    println!("split at index {} — never shuffle across this boundary", split);
}

## Stationarity — is the series safe to model?

Most classical forecasters (ARIMA in the [forecasting chapter](forecasting.ipynb))
assume the series is **stationary**: its mean and variance don't drift over time.
Our series has an upward **trend**, so it isn't — and the standard check is the
**Augmented Dickey-Fuller** test. [`chronos-ts`](https://crates.io/crates/chronos-ts)
provides `adf_test` (null hypothesis: a unit root, i.e. *non*-stationary — a small
p-value means we reject it and call the series stationary) and `estimate_d`, which
reports how many times to **difference** the series to get there — that's the "I"
(Integrated) in ARIMA.

In [ ]:
:dep chronos-ts = "0.1"
:dep ndarray = { version = "0.15" }
use chronos_ts::{adf_test, estimate_d};
use ndarray::Array1;

let s = Array1::from_vec(series.clone());
// First difference: y[t] - y[t-1], which removes a linear trend.
let diff = Array1::from_vec((1..series.len()).map(|i| series[i] - series[i - 1]).collect::<Vec<f64>>());
let level = adf_test(&s, None);
let differenced = adf_test(&diff, None);
println!("ADF on the level:          stat = {:>7.3}   p = {:.3}   -> {}", level.stat, level.p_value, if level.p_value < 0.05 { "stationary" } else { "NON-stationary" });
println!("ADF on the 1st difference: stat = {:>7.3}   p = {:.3}   -> {}", differenced.stat, differenced.p_value, if differenced.p_value < 0.05 { "stationary" } else { "NON-stationary" });
println!("estimate_d: {} difference(s) needed to reach stationarity", estimate_d(&s, 2, 0.05));

## Rolling-origin cross-validation

Instead of random folds, grow the training window forward in time and always
test on the *next* chunk. Each fold trains only on data that precedes its test
set — no leakage:

In [ ]:
{
    let initial = 28;   // first training window
    let horizon = 7;    // test one week ahead each fold
    let mut origin = initial;
    let mut fold = 1;
    while origin + horizon <= series.len() {
        println!("fold {}: train [0..{}]  ->  test [{}..{}]", fold, origin, origin, origin + horizon);
        origin += horizon;
        fold += 1;
    }
}

## Lag features & rolling statistics

To let an ordinary regressor use the past, engineer **lag** columns (the value
*k* steps ago) and **rolling** statistics (a moving average). This ties back to
the [ETL chapter's](../01c-etl/data-preparation.ipynb) feature engineering, now
with time awareness:

In [ ]:
{
    // For each day t (from 7 on), build [value, lag-1, lag-7, rolling-mean-3].
    println!("{:>3}  {:>7}  {:>7}  {:>7}  {:>10}", "t", "value", "lag1", "lag7", "roll_mean3");
    for t in 7..12 {
        let value = series[t];
        let lag1 = series[t - 1];
        let lag7 = series[t - 7];
        let roll_mean3 = (series[t - 1] + series[t - 2] + series[t - 3]) / 3.0;
        println!("{:>3}  {:>7.2}  {:>7.2}  {:>7.2}  {:>10.2}", t, value, lag1, lag7, roll_mean3);
    }
}

Those engineered columns turn a forecasting problem into an ordinary supervised
one. Next: [forecasting](forecasting.ipynb) — a naive baseline, a real model
(`augurs` MSTL), and time-series accuracy metrics.